# Machine Learning Basics

Machine learning (ML) is a branch of artificial intelligence that enables computers to identify patterns and make predictions from data with minimal human intervention. In drug discovery, ML algorithms can analyze vast datasets of chemical structures, biological activities, and pharmacokinetic properties to predict which compounds are likely to be effective drugs. Applications include virtual screening of compound libraries, prediction of drug-target interactions, identification of potential toxicity, and optimization of lead compounds through structure-activity relationship (SAR) modeling. By accelerating these processes, ML helps reduce the time and cost associated with bringing new drugs to market.

In this module you will build binary classification models that predict activity/inactivity of small molecules against human aromatase using supervised learning methods, and evaluate the performance of the developed models using performance measures.

<div class="alert alert-block alert-info">
<h2>Module Learning Objectives</h2>

* Identify and address data quality issues to prepare chemical and biological data for machine learning.

* Build and interpret binary classification models for prediction of activities at a target
    * Naïve Bayes
    * Decision tree
    * Random forest
* Explore the Scikit-learn open-source machine learning Python library
* Develop further Pandas skills to clean downloaded data
* Review PubChem's PUG REST Web Interface
* Practice code from previous notebooks


**Reading links**

* [What are Naïve Bayes classifiers?](https://www.ibm.com/think/topics/naive-bayes)
* [What is a decision tree?](https://www.ibm.com/think/topics/decision-trees)
    * [What is hyperparameter tuning?](https://www.ibm.com/think/topics/hyperparameter-tuning)
* [What is random forest?](https://www.ibm.com/think/topics/random-forest)


<div class="alert alert-block alert-info">

# Part 1: Obtaining and Cleaning Data

## 1. Import bioactivity data from PubChem

In this notebook, we will develop a prediction model for small molecule's activity tested against human aromatase (https://pubchem.ncbi.nlm.nih.gov/protein/EAW77416), an enzyme encoded by the CYP19A1 gene (https://pubchem.ncbi.nlm.nih.gov/gene/1588). Aromatase converts androgens into estrogens, so it plays an important role in controlling the balance between these two classes of steroid hormones.

Aromatase is an important target when studying endocrine disrupting chemicals (EDCs). They are chemicals that interfere with the biosynthesis and normal functions of steroid hormones including estrogen and androgen in the body. Aromatase catalyzes the conversion of androgen to estrogen and plays a key role in maintaining the androgen and estrogen balance in many of the EDC-sensitive organs. A chemical that changes aromatase activity could potentially disrupt normal endocrine function.

For this model, we will use the Tox21 bioassay data for human aromatase, archived in PubChem (https://pubchem.ncbi.nlm.nih.gov/bioassay/743139). This is a high throughput assay to identify aromatase inhibitors (**antagonists**). Our model will use each molecule’s structure, represented by molecular fingerprints, to predict whether that molecule is active in the aromatase assay. The data can be downloaded from the PubChem bioassay page or loaded directly into a data frame using the code below.

In [ ]:
import pandas as pd
import numpy as np

url = 'https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download&record_type=datatable&actvty=all&response_type=save&aid=743139'
df_raw = pd.read_csv(url)

In [ ]:
df_raw.shape

In [ ]:
df_raw.to_csv("raw_743139.csv")  # saves the data in case we need it again and don't have PubChem access.

In [ ]:
df_raw.head(7)  # gives the first 7 rows of the dataframe
#df_raw.tail(7) # gives the last 7 rows of the dataframe

Data cleaning is a critical step in our workflow that ensures the quality, consistency, and reliability of a dataset before we build a model. Raw data often contains missing values, duplicates, inconsistent formatting, or mislabeled entries that can lead to incorrect conclusions or poorly performing models. By identifying and resolving these issues at the start of our model design, we create a dataset that accurately reflects the information needed for meaningful interpretation and predictive modeling. 

Looking at our imported data, lines 0-2 provide the descriptions for each column (data type, descriptions, units, etc).  These rows need be removed. We will do so with a **slice operation** to select only those rows that contain the data we will use in the model.

In [ ]:
df_raw = df_raw[3:]  #the [3:] indicates a slice opperation in pandas dataframes. It selects all rows starting
                     #at position 3 and continues to the end.
df_raw.head(5)

In [ ]:
df_raw.shape

The column names in this data frame contain white spaces and special characters.  For simplicity, let's rename the columns (no spaces or special characters except for the "_" character.)

In [ ]:
df_raw.columns #print out the column names. 

In [ ]:
col_names_map = {'PUBCHEM_RESULT_TAG' : 'pc_result_tag', 
                 'PUBCHEM_SID' : 'sid', 
                 'PUBCHEM_CID' : 'cid',
                 'PUBCHEM_ACTIVITY_OUTCOME' : 'activity_outcome', 
                 'PUBCHEM_ACTIVITY_SCORE' : 'activity_score',
                 'PUBCHEM_ACTIVITY_URL' : 'activity_url', 
                 'PUBCHEM_ASSAYDATA_COMMENT' : 'assay_data_comment', 
                 'Activity Summary' : 'activity_summary',
                 'Antagonist Activity' : 'antagonist_activity', 
                 'Antagonist Potency (uM)' : 'antagonist_potency', 
                 'Antagonist Efficacy (%)' : 'antagonist_efficacy',
                 'Viability Activity' : 'viability_activity', 
                 'Viability Potency (uM)' : 'viability_potency',
                 'Viability Efficacy (%)' : 'viability_efficacy', 
                 'Sample Source' : 'sample_source' }

In [ ]:
df_raw = df_raw.rename(columns = col_names_map)
df_raw.columns

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong>

1) Why is it important to rename the column names without spaces or special characters?

2) In the code cell below, display the first 10 rows of the dataframe to indicate that the column names have been remapped.



In [ ]:
# Write your code here for question 2


## 2. Check the number of compounds for each activity group

Since our goal is to develop a model that classifies small molecules based on their activity against a target, we first need to understand the the structure of our data. One important part of the dataset is the **activity class** assigned to each tested compounds In this dataframe, activity information is found in the `activity_outcome` and `activity_summary` columns of the dataframe.

To explore these categories, we use the `groupby()` method. This method organizes the dataframe into groups based on the values in one or more columns. We can then apply a calculation to each group separately. In the code cell below, we group the dataframe by `activity_outcome` and use the `count()` method to determine how many entries fall into each activity category.

In [ ]:
df_raw.groupby(['activity_outcome']).count()

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answers in the next raw cell.

1) How many actives, inactives and inconclusive compounds are there in the dataset based on the `activity_outcome` column?
2) Is this the same value as your original dataframe after removal of description lines?
3) Scroll through the rest of the data. Compare these numbers to other columns such as CID and SMILES. What problems might arise if these counts are not consistent?


If we group by both `activity_outcome` and `activity_summary` we get more insight about the data. Particularly, it reveals information that will indicate that there are subcategories for `Inconclusive`, giving us a better understanding of how thse results are classified.

In [ ]:
df_raw.groupby(['activity_outcome','activity_summary']).count()

In [ ]:
print('there are',len(df_raw),'compounds in the dataframe that are defined as active antagonist, inactive, or inconclusive.')

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answers in the next raw cell.

1) How many subcategories are found in the `Inconclusive` `activity_outcome` category?
2) You previously found there are 2545 `Inconclusive` molecules. Is that still true in this grouping? Explain how you arrived at your decision.
3) What is the largest subcategory of inconclusive activity outcomes?
4) Why do you think a cytotoxic molecule would be labeled as inconclusive either as an agonist or an antagonist?

We now see in the **activity_summary** column that the inconclusive compounds are further classified into subclasses. These are:

- active agonist
- inconclusive
- inconclusive agonist
- inconclusive antagonist
- inconclusive agonist (cytotoxic)
- inconclusive antagonist (cytotoxic)

As implied in the title of this [PubChem assay record (AID 743139)](https://pubchem.ncbi.nlm.nih.gov/bioassay/743139), the goal of this assay is to identify **aromatase inhibitors**. Accordingly, all compounds labeled as active antagonists in the **activity_summary** column were marked as active in the **activity_outcome** column.

However, the assay also identified 612 **active agonists**, which are labeled as inconclusive in the **activity_outcome** column. This reflects the original submitter's specific classification criteria: only antagonists were considered **"active"** for the purposes of identifying inhibitors.

In this context, **inactive** compounds are those that show neither agonist nor antagonist activity.

It's important to recognize that the definitions of **“active”** and **“inactive”** depend on how the assay was designed and how its results were interpreted by the data submitter. For the purpose of this notebook (which aims to build a binary classifier to predict whether a compound is active or inactive) we will redefine these labels for consistency:

* **Active**: Any compound that alters the activity of the target, either by increasing (agonist) or decreasing (antagonist) its function. This includes all compounds labeled as **active antagonists** or **active agonists** in the **activity_summary** column.

* **Inactive**: Compounds that do not change the activity of the target, corresponding to those labeled inactive in the **activity_summary** column.

In short, we are treating *<u>any molecule</u>* that interacts with the enzyme as **active**, and those that do not as **inactive**. This operational definition is critical for building a model that accurately classifies compound-target interactions.

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the the following raw cell.

Imagine we build our model using only compounds labeled as active antagonists as the "active" class. What might be the consequences of this decision for the type of model we create? Could this be useful in some cases? What would it miss?

In [ ]:
Answer:



## 3. Select active/inactive compounds for model building

Based on our model objectives, we want to select only the active and inactive compounds from the data frame (that is, active agonists, active antagonists, and inactives based on the "activity summary" column).

In [ ]:
# create a new dataframe with active agonists, active antagnonists and inactives
# remove any molecules that are inconclusive
df = df_raw[ (df_raw['activity_summary'] == 'active agonist' ) | 
             (df_raw['activity_summary'] == 'active antagonist' ) |
             (df_raw['activity_summary'] == 'inactive' ) ]

print("The number of total molecules that are active or inactive is:", len(df))

Since we will be obtaining structural data from PubChem, it's important to examine how many unique PubChem Substance IDs (SIDs) and Compound IDs (CIDs) are present in our dataset.

Recall the distinction:

* **Substances (SIDs)** represent depositor-submitted records. Multiple SIDs may refer to the same chemical structure if submitted by different sources.

* **Compounds (CIDs)** are standardized, unique chemical structures derived by PubChem from submitted substances through a process of structure normalization and deduplication.

Understanding how many unique SIDs and CIDs we have will help us assess redundancy and ensure we're working with non-duplicated chemical structures when building models or visualizing data.

In [ ]:
# identify Compounds IDs and Substance IDs. 
print('total number of unique Substance IDs=',len(df['sid'].unique()))
print('total number of unique Compound IDs= ',len(df['cid'].unique()))

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the the following raw cell.

1) Which data SIDs or CIDs matches the number of active and inactive molecules in the new dataframe,df?
2) Why might there be fewer unique CIDs than SIDs in our new dataframe, df?

This data indicates two important points:

1) The number of unique Substance IDs matches the number of active and inactive molecules in the dataset. 

2) Not all substances have an associated Compound ID (CID), meaning some structures have not been standardized by PubChem.

Since our goal is to build a model that uses chemical structure to predict biological activity, we must remove substances without CIDs as these lack standardized structural information necessary for modeling.

Additionally, we should be aware that multiple substances can map to the same compound. This happens when the same chemical structure is submitted by different sources or tested under different conditions. As a result, one CID might be associated with conflicting activity outcomes, for example, one sample labeled as an active agonist and another as inactive. To maintain consistency in the training data, we will remove compounds with conflicting activities.

By cleaning our data based on CID and activity type, we ensure the model learns from reliable, unambiguous data where each chemical structure is uniquely associated with a single activity classification.

### 3a. Drop substances without associated CIDs.

First, check if there are subtances without associated CIDs. We will used the `isna()` method to identify all rows that are missing values (not available). 

**Chaining** refers to calling multiple methods one after another on a pandas object. We can **"chain"** this with `.sum()` to count how many are missing. 

In [ ]:
df.isna().sum()  

<div class="alert alert-block alert-warning">

<strong>Check your understanding</strong> Write your answer in the the following raw cell.
1) How many records are in the dataset? (look at shape.df or the value of len(df) above)
2) How many records lack an associated CID? 
3) How many records lack an activty URL? What percent of the dataset does this represent?

The records without CID are not going to provide us with standardized structural information, so we will remove them from the dataframe. 

This is accomplished by using the `.dropna()` method. Since we only want to remove rows where the CID is missing, we can use the `subset` parameter to specify we are targeting only those NA values in the **'cid'** column. 

Look at the output of the previous cell block. If we did the `dropna()` method without indicating which subset we are dropping, we could lose all of our data because the **activity_url** is null for every row in the dataframe!

In [ ]:
print(len(df))   #in previous we saw 138 rows without CID
df = df.dropna( subset=['cid'] )
print(len(df))   #this valuse should be 138 fewer

In [ ]:
# identify Compounds IDs and Substance IDs. 
print('total number of Substance IDs=',len(df['sid'].unique()))
print('total number of Compound IDs= ',len(df['cid'].unique()))

Check if the NULL values disappeared in the **cid** column. The value should be 0 for that column now.

In [ ]:
df.isna().sum()

### 3b. Remove CIDs with conflicting activities

This code identifies unique CIDs and checks whether multiple values exist in the **activity_summary column**. If conflicting activity summaries are found, the code stores the CID in the `cid_conflict` list and and records the corresponding row indices in `idx_conflict`. Finally, the total number of CIDs with conflicts and total number of rows are output.

In [ ]:
cid_conflict = [] # list to store CIDs with conflicting activities
idx_conflict = [] # list to store indices of rows with conflicting activities

for mycid in df['cid'].unique() : # iterate over each unique Compound ID
    
    outcomes = df[ df.cid == mycid ].activity_summary.unique() #
    
    if len(outcomes) > 1 : # if there are multiple unique activity summaries for this CID
        
        idx_tmp = df.index[ df.cid == mycid ].tolist() # get the indices of these rows
        idx_conflict.extend(idx_tmp) # add these indices to the conflict list
        cid_conflict.append(mycid) # # add the CID to the conflict list

print("#", len(cid_conflict), "CIDs with conflicting activities [associated with", len(idx_conflict), "rows (SIDs).]")
#print(idx_conflict)

To examine which CIDs have conflicting activity data, we can display a portion of the dataframe using the `.loc[]` method. This allows us to select specific rows. In this case, we used the ones listed in `idx_conflict`, which correspond to compounds with inconsistent activity summaries. We'll chain with `.head(10)` to view just the first 10 of these rows for quick inspection.

This helps us verify the nature of the conflicts and better understand why these entries need to be removed before modeling.

In [ ]:
df.loc[idx_conflict,:].head(10)

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the the following raw cell.
<br><br>

Examine the portion of the dataframe above. Why should CID 16043 be removed from the analysis?

In [ ]:
# Drop the IDs that have conflicting data
df = df.drop(idx_conflict)

In [ ]:
# After removing the conflicting data, we can check the counts of each activity summary again
df.groupby('activity_summary').count()

In [ ]:
# identify Compounds IDs and Substance IDs. 
print('total number of Substance IDs=',len(df['sid'].unique()))
print('total number of Compound IDs= ',len(df['cid'].unique()))

### 3c. Remove redundant data

The code cells in section 3b. do not remove compounds tested multiple times if the testing results are consistent [e.g., active agonist in all samples (substances)].  The rows corresponding to these compounds are redundant, so we want remove them except for only one row for each compound.

To illustrate this, let's create a sorted dataframe to see some of the redundancies.

In [ ]:
sorted_df = df.sort_values(by=['cid','sid','activity_summary'],
                           ascending=[True,True,True])
sorted_df.head(10)

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the the following raw cell.
    
Explore the above displayed sorted dataframe. Identify any CIDs that are repeated, and if they have different <b>SID</b> or <b>activity_outcome</b> values.

In [ ]:
Answer:



We will use the `drop_duplicates()` method in pandas to remove any duplicate rows in our dataframe. We can use the `subset` parameter to signify which column to consider when identifying duplicates. In our case, we need unique identifiers to get structural data, so we use **'cid'**.

In [ ]:
df = df.drop_duplicates(subset='cid')  # remove duplicate rows except for the first occurring row.
print('total number of Substance IDs=',len(df['sid'].unique()))
print('total number of Compound IDs= ',len(df['cid'].unique()))

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answer in the the following raw cell.
    
    
Why are the total of SIDs equal to CIDs at this point?

### 3d. Adding "numeric" activity classes

In general, machine learning algorithms require both inputs and outputs to be in numerical form. 

In our case the input we will be using is molecular structure. We have been working to clean our data to ensure we have valid CIDs that we can use to retrieve SMILES strings and subsequently calculate binary fingerprints as input features.
 
The output feature of biological activity is currently stored as text labels as `active` or `inactive`. To make this data suitable for modeling, we will create a new column called `activity` that assigns these labels as numeric values:

- `1` for actives (including both active agonists and antagonists)
- `0` for inactives

We combine the two types of active compounds into a single class because our goal is to train a binary classifier that distinguishes between compounds that interact with the target (active) and those that do not (inactive).

In [ ]:
df['activity'] = [ 0 if x == 'inactive' else 1 for x in df['activity_summary'] ]

Check if the new column 'activity' is added to (the end of) the data frame.

In [ ]:
df.head(3) #new column is added to end of column list, so scroll right in the resulting cell output.

Double-check the count of active/inactive compounds.

In [ ]:
df.groupby('activity_summary').count()

In [ ]:
df.groupby('activity').count() 

In [ ]:
Answer:


### 3e. Create a smaller data frame that only contains CIDs and activities.

Let's create a smaller data frame that only contains CIDs and activities.  This data frame will be merged with a data frame containing molecular fingerprint information.

In [ ]:
df_activity = df[['cid','activity','PUBCHEM_EXT_DATASOURCE_SMILES']]

In [ ]:
df_activity.head(5)

In [ ]:
df_activity.shape

## 4. Download structure information for each compound from PubChem

While we could have used the SMILES that were already in our dataframe, the label was PUBCHEM_EXT_DATASOURCE_SMILES indicating that the SMILES may have been provided by the submitter, and not cannonical.  To ensure we have quality data, we will retrieve SMILES from PubChem.

Notice also in the previous dataframe, our CID values all had decimals. That is because when we brought in the data to the dataframe, pandas inferred they data type as `float64`. We can confirm this and create a new list where all the data is of type `int`.

In [ ]:
print("CIDs are stored in original df dataframe as", df['cid'].dtype)
print("CIDs are stored in df_activity dataframe as", df_activity['cid'].dtype)
cids = df.cid.astype(int).tolist()
#print(cids)   #this prints the list of CIDs as a debugging exercise to see if all cids in df are in the list
print("# of CIDs in cids list =",len(cids)) #should equal rows in df

Now that we have a list of CIDs as integers, we can retrive the SMILES from PubChem using the PUG REST API. 

Once we have retrieved the SMILES we will store them and their respective CIDs into a new `df_smiles` dataframe.

In [ ]:
chunk_size = 200
num_cids = len(cids)

if num_cids % chunk_size == 0 :
    num_chunks = int( num_cids / chunk_size )
else :
    num_chunks = int( num_cids / chunk_size ) + 1

print("# CIDs = ", num_cids)
print("# CID Chunks = ", num_chunks, "(chunked by ", chunk_size, ")")

In [ ]:
# This section of code takes about 1 minute

import time
import requests
from io import StringIO

df_smiles = pd.DataFrame()
list_dfs = []  # temporary list of data frames

for i in range(0, num_chunks) :
    
    idx1 = chunk_size * i
    idx2 = chunk_size * (i + 1)
    cidstr = ",".join( str(x) for x in cids[idx1:idx2] )
    url = ('https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/' + cidstr + '/property/SMILES/TXT')
    res = requests.get(url)
    data = pd.read_csv( StringIO(res.text), header=None, names=['smiles'] )
    list_dfs.append(data)
    
    time.sleep(1)  #increase from 0.2 to see if that helps. Was getting timeout at 25 chunks
    
    if ( i % 5 == 0 ) :
        print("Processing Chunk ", i)   

#    if ( i == 2 ) : break  #- for debugging

df_smiles = pd.concat(list_dfs,ignore_index=True)
df_smiles[ 'cid' ] = cids
df_smiles.head(5)

In [ ]:
# make sure the number of rows in df_smiles is equal to the number of unique CIDs we had previously
print("Number of CIDs in df_smiles:", len(df_smiles))
print("Number of unique CIDs in original df:", len(df['cid'].unique()))

print("The number of CIDs in df_smiles is equal to the number of unique CIDs in the original df:",
       len(df_smiles) == len(df['cid'].unique()))

In [ ]:
# reorder columns to have 'cid' first, then 'smiles'
df_smiles = df_smiles[['cid','smiles']]
df_smiles.head(5)

Now we have two dataframes of our cleaned data:
- `df_smiles` which contains our CIDs and cannonical SMILES from PubChem 
- `df_activity` which contains our CIDs and activity data (agonist and antagonist = active = 1, inactive = 0)

Let's save our data so we have a starting point for the next part and not have to regenerate all the data.

In [ ]:
df_smiles.to_csv('AID743139_SMILES_cids.csv', index=False)
df_activity.to_csv('AID743139_activity_cids.csv', index= False)

<div class="alert alert-block alert-warning">
<strong>Check your understanding</strong> Write your answers in the following raw cell.

1) When we requested the SMILES data from PubChem through PUG REST, why did we chunk our data and use time.sleep(.2)?

2) Briefly explain the steps and importance for cleaning the data in the notebook thus far. 


In [ ]:
Answers:

1)

2)